In [ ]:
# COMP541 organized folder path helper
from pathlib import Path
import os

def _comp541_phase_dir():
    cwd = Path.cwd().resolve()
    if cwd.name.lower() == 'code':
        return cwd.parent
    if (cwd / 'Input').exists() and (cwd / 'Output').exists():
        return cwd
    if (cwd.parent / 'Input').exists() and (cwd.parent / 'Output').exists():
        return cwd.parent
    # Fallback for the intended layout: notebook/script is inside a Code folder.
    return cwd.parent if cwd.name.lower() == 'code' else cwd

PHASE_DIR = _comp541_phase_dir()
INPUT_DIR = PHASE_DIR / 'Input'
OUTPUT_DIR = PHASE_DIR / 'Output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('INPUT_DIR :', INPUT_DIR.resolve())
print('OUTPUT_DIR:', OUTPUT_DIR.resolve())


# **Traveler Satisfaction from Review Sentiment** (Proposal Goal 3)

*COMP 541 Data Mining · Group 1.*

The proposal calls for a **sentiment model over the TripAdvisor hotel-review text**,
aggregated into a **destination-level satisfaction score** that feeds the Budget Value Index.
This notebook produces that score.

**Engine.** The proposal's default is DistilBERT on a Colab GPU, with a fallback to
a lightweight pre-trained model when GPU resources are unavailable. This local run uses
**VADER** (CPU-reproducible); a swappable `score_distilbert` path runs unchanged on Colab.

**Output.** `destination_satisfaction.csv` — one row per reviewed city, satisfaction in
**[-1, 1]**, ready to merge into the modeling notebook.

#### **Imports & sentiment engine**

In [1]:
import os
import re
import unicodedata

import numpy as np
import pandas as pd

import nltk
try:
    from nltk.sentiment import SentimentIntensityAnalyzer
    _sia = SentimentIntensityAnalyzer()
except LookupError:
    nltk.download("vader_lexicon")
    from nltk.sentiment import SentimentIntensityAnalyzer
    _sia = SentimentIntensityAnalyzer()


def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)


def score_vader(texts):
    '''Pre-trained lexicon sentiment -> compound score in [-1, 1] per text.'''
    return np.array([_sia.polarity_scores(str(t))["compound"] for t in texts])


def score_distilbert(texts, batch_size=64):
    '''Transformer path for Colab GPU (proposal default). Maps SST-2 prob to [-1, 1].
    Falls back with a clear message if torch/transformers are unavailable locally.'''
    from transformers import pipeline  # requires torch + a working DLL/GPU
    clf = pipeline("sentiment-analysis",
                   model="distilbert-base-uncased-finetuned-sst-2-english",
                   truncation=True, batch_size=batch_size, device=0)
    out = clf(list(map(str, texts)))
    return np.array([(r["score"] if r["label"] == "POSITIVE" else -r["score"]) for r in out])


# Choose the engine: VADER locally, DistilBERT on Colab GPU.
SENTIMENT_ENGINE = "vader"   # set to "distilbert" on Colab with a GPU runtime
score_fn = score_vader if SENTIMENT_ENGINE == "vader" else score_distilbert
print("sentiment engine:", SENTIMENT_ENGINE)

sentiment engine: vader


#### **Locate the TripAdvisor file & join-key helper**

The review file is large (~846 MB, ~879k reviews across 26 US cities) and not UTF-8, so we
stream it in chunks with `latin-1` (maps all bytes; `make_key` strips non-ASCII anyway).

In [2]:
# Organized folder paths
TRIP_CSV = INPUT_DIR / "Tripadvisor cleaned.csv"
OUTPUT_DIR = str(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not TRIP_CSV.exists():
    raise FileNotFoundError(f"Could not find TripAdvisor input CSV: {TRIP_CSV}")

print("TripAdvisor:", TRIP_CSV)
print("OUTPUT_DIR :", OUTPUT_DIR)

CITY_ALIASES = {"new york": "new york city", "antwerp": "antwerpen", "quebec city": "quebec"}


def make_key(text, alias=None):
    if pd.isna(text):
        return np.nan
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[\"'{}]", "", text).lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    if alias and text in alias:
        text = alias[text]
    return text

TripAdvisor: C:\Users\jxs75\Documents\GitHub\Data Mining\data\cleaned\TripAdvisor cleaned.csv
OUTPUT_DIR : C:\Users\jxs75\Documents\GitHub\Data Mining\data\cleaned\alternate integration output


#### **Stream the reviews and take a capped sample per city**

To bound compute we keep up to `CAP_PER_CITY` reviews per city (a stable sample for a mean).

In [3]:
CAP_PER_CITY = 500
USE_COLS = ["City", "State", "text", "Overall", "Value"]
samples = {}   # city_key -> list of (text, overall, value)

section("STREAMING REVIEWS")
total = 0
for chunk in pd.read_csv(TRIP_CSV, encoding="latin-1", usecols=USE_COLS,
                         chunksize=200000, on_bad_lines="skip"):
    total += len(chunk)
    chunk["ck"] = chunk["City"].map(lambda c: make_key(c, CITY_ALIASES))
    chunk = chunk.dropna(subset=["ck", "text"])
    for ck, grp in chunk.groupby("ck"):
        bucket = samples.setdefault(ck, [])
        if len(bucket) >= CAP_PER_CITY:
            continue
        need = CAP_PER_CITY - len(bucket)
        bucket.extend(grp[["text", "Overall", "Value"]].head(need).itertuples(index=False, name=None))

print(f"scanned {total:,} review rows | cities sampled: {len(samples)}")
print("sample sizes:", {k: len(v) for k, v in sorted(samples.items())[:8]}, "...")


STREAMING REVIEWS
scanned 878,949 review rows | cities sampled: 26
sample sizes: {'austin': 500, 'baltimore': 500, 'boston': 500, 'charlotte': 500, 'chicago': 500, 'columbus': 500, 'dallas': 500, 'denver': 500} ...


#### **Run sentiment & aggregate to a destination satisfaction score**

In [4]:
rows = []
for ck, recs in samples.items():
    if ck in ("locality", ""):  # junk city label in the source
        continue
    texts = [r[0] for r in recs]
    comp = score_fn(texts)                       # per-review sentiment in [-1, 1]
    overall = pd.to_numeric([r[1] for r in recs], errors="coerce")
    value = pd.to_numeric([r[2] for r in recs], errors="coerce")
    rows.append({
        "city_key": ck,
        "country_key": "united states",
        "n_reviews_scored": len(texts),
        "satisfaction_compound": round(float(np.mean(comp)), 4),     # [-1, 1]
        "satisfaction_0_1": round(float((np.mean(comp) + 1) / 2), 4), # rescaled for BVI
        "mean_overall_rating": round(float(np.nanmean(overall)), 2),
        "mean_value_rating": round(float(np.nanmean(value)), 2),
    })

sat = pd.DataFrame(rows).sort_values("satisfaction_compound", ascending=False).reset_index(drop=True)
# NLP validity (proposal §11): scores in [-1,1], zero nulls
assert sat["satisfaction_compound"].between(-1, 1).all()
assert sat["satisfaction_compound"].notna().all()
section("DESTINATION SATISFACTION (sentiment-derived)")
print(sat.to_string(index=False))


DESTINATION SATISFACTION (sentiment-derived)
     city_key   country_key  n_reviews_scored  satisfaction_compound  satisfaction_0_1  mean_overall_rating  mean_value_rating
new york city united states               500                 0.9163            0.9581                 4.59               4.36
      phoenix united states               500                 0.8982            0.9491                 4.72               4.43
san francisco united states               500                 0.8891            0.9445                 4.45               4.24
washington dc united states               500                 0.8853            0.9427                 4.49               4.21
    san diego united states               500                 0.8701            0.9351                 4.44               4.31
  san antonio united states               500                 0.8701            0.9350                 4.53               4.47
 philadelphia united states               500                 0.8

#### **Cross-check & export**

Sanity check: sentiment should correlate with the raw star ratings (higher stars →
more positive text). Then export for the modeling notebook.

In [5]:
r = sat["satisfaction_compound"].corr(sat["mean_overall_rating"])
print(f"corr(sentiment, mean Overall rating) = {r:.3f}  "
      f"({'consistent' if r > 0.3 else 'weak - inspect'})")

out_path = os.path.join(OUTPUT_DIR, "destination_satisfaction.csv")
sat.to_csv(out_path, index=False)
section("EXPORTED")
print(out_path, "->", sat.shape)
print("\nThis merges into Data_Modeling.ipynb on (city_key, country_key) as the proposal's "
      "'satisfaction' BVI ingredient (bonus where present).")

corr(sentiment, mean Overall rating) = 0.958  (consistent)

EXPORTED
C:\Users\jxs75\Documents\GitHub\Data Mining\data\cleaned\alternate integration output\destination_satisfaction.csv -> (25, 7)

This merges into Data_Modeling.ipynb on (city_key, country_key) as the proposal's 'satisfaction' BVI ingredient (bonus where present).
